In [ ]:
import pandas as pd
import numpy as np 
import datetime 
import warnings

This program cleans a COPQ extract from COPQ Spotfire dashboard. 

It adds in Perfect order fulfillment attributes from TOAD table POF_CUST_OTD as well as categorizes 2.3 or 2.4 gradings to lines extracted.

TOAD:

select *

from

EID.POF_CUST_OTD

where 

TARGET_DATE > TO_DATE('01/01/2022', 'MM/DD/YYYY')
;

In [111]:
#import data
COPQ_df = pd.read_excel("COPQ.xlsx")
##SQL_df = pd.read_csv("SQL_.csv",delimiter='|')
##NOTBAT_df = pd.read_csv("NOTIFICATION_BATCH.csv",delimiter='|')

In [112]:
# Convert to a SET of strings for high-performance filtering
COPQ_set = COPQ_df.dropna(subset=['SALES_ORDER_NUM'])
COPQ_set['SALES_ORDER_NUM'] = COPQ_set['SALES_ORDER_NUM'].astype(int)
COPQ_set = set(COPQ_set['SALES_ORDER_NUM'].astype(str).unique())

filtered_chunks = []

# Process in chunks
for chunk in pd.read_csv("POF_CUST_OTD.csv", delimiter='|', chunksize=100000, dtype=str):
    # .isin() works efficiently with sets
    matches = chunk[chunk['sales_order'].isin(COPQ_set)]
    
    if not matches.empty:
        filtered_chunks.append(matches)

final_df = pd.concat(filtered_chunks, ignore_index=True)

C:\Users\ti290f\AppData\Local\Temp\ipykernel_37452\2292003692.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  COPQ_set['SALES_ORDER_NUM'] = COPQ_set['SALES_ORDER_NUM'].astype(int)


In [113]:
final_df_2 = final_df.copy()

#Drop duplicate SO values and construct key
final_df_2['key'] = final_df_2['sales_order'] + '.' +final_df_2['so_line']
SO_key_unique = final_df_2.drop_duplicates(subset=['key'], keep='first')

#drop na in sales lines and create key within COPQ
###COPQ_df['SALES_ORDER_NUM'] = pd.to_numeric(COPQ_df['SALES_ORDER_NUM'], errors='coerce')

COPQ_df_with_SO_LN = COPQ_df.dropna(subset=['SALES_ORDER_LN'])
COPQ_df_with_SO_LN['SALES_ORDER_NUM'] = COPQ_df_with_SO_LN['SALES_ORDER_NUM'].astype(int).copy()
COPQ_df_with_SO_LN['SALES_ORDER_LN'] = COPQ_df_with_SO_LN['SALES_ORDER_LN'].astype(int).copy()
COPQ_df_with_SO_LN['SALES_ORDER_NUM'] = COPQ_df_with_SO_LN['SALES_ORDER_NUM'].astype(str).copy()
COPQ_df_with_SO_LN['SALES_ORDER_LN'] = COPQ_df_with_SO_LN['SALES_ORDER_LN'].astype(str).copy()

COPQ_df_with_SO_LN['key'] = COPQ_df_with_SO_LN['SALES_ORDER_NUM'] + '.' + COPQ_df_with_SO_LN['SALES_ORDER_LN']

#merge files
COPQ_df_with_SO_LN = COPQ_df_with_SO_LN.merge(SO_key_unique[['key','target_date','on_time','in_full','make_miss']], left_on='key',right_on='key', how='left')

C:\Users\ti290f\AppData\Local\Temp\ipykernel_37452\3849492266.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  COPQ_df_with_SO_LN['SALES_ORDER_NUM'] = COPQ_df_with_SO_LN['SALES_ORDER_NUM'].astype(int).copy()
C:\Users\ti290f\AppData\Local\Temp\ipykernel_37452\3849492266.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  COPQ_df_with_SO_LN['SALES_ORDER_LN'] = COPQ_df_with_SO_LN['SALES_ORDER_LN'].astype(int).copy()
C:\Users\ti290f\AppData\Local\Temp\ipykernel_37452\3849492266.py:13: SettingWithCopyWarnin

In [114]:
defect_list=('BoxLabelsIncorrect/Missing',
'BrokenKit',
'CannotVerifyBATCH/LOT',
'CannotVerifyPARTNUMBER',
'Concealed',
'DamagedPart',
'DoesnotmeetDrawingSpecifications',
'Doesnotmeetrequirements',
'Expired',
'IncorrectBATCH/LOT',
'ManufacturingDefect',
'MFGLabelvsPhysicalPart(PN,SN)',
'MissingBATCH/LOT',
'Over',
'PackQuantity',
'Packagingnotacceptable',
'PackagingUnacceptable',
'Under',
'VendorLabelIncorrect',
'WrongPart')
COPQ_df_with_SO_LN['SCOR_2.3'] = (~COPQ_df_with_SO_LN['Defect Desc'].isin(defect_list)).astype(int)
COPQ_df_with_SO_LN['SCOR_2.4'] = COPQ_df_with_SO_LN['Defect Desc'].isin(defect_list).astype(int)

In [119]:
COPQ_df_with_SO_LN['target_date'] = pd.to_datetime(COPQ_df_with_SO_LN['target_date'], errors='coerce')
COPQ_df_with_SO_LN.to_excel('COPQ_df_with_SO_LN.xlsx', index=False)